# Bit-flip rate vs pump strength, with Krylov-Schur

Sweeps `eps_p` at fixed `alpha_sq`, rescaling `kappa_b` as
`sin(eps_p)/sin(eps_p_init) * kappa_b_init` to hold the adiabatic ratio
`kappa_b / g` constant -- the sweep of `tests/test_sweep_eps_p.ipynb`, with
`KrylovSchurLindblad` (thick-restarted Arnoldi on the one-period propagator,
`docs/krylov_schur.md`) in place of the Chebyshev-filtered `ChebAr` pipeline.

**This sweep is sequential, unlike the `alpha_sq` one.** Each point is
warm-started from the previous point's Ritz vector, re-expressed in the new
eigenbasis (`transform_vectorized_state`) because the interaction-frame basis
moves with `eps_p`. Only the first point runs cold, with the larger restart
size `M_FIRST`. So the chain runs one GPU task per point, in order, rather than
fanning out.

Set `WARM_START = False` to break the chain deliberately: every point then runs
cold and independent, and they all run in parallel -- more GPU-seconds, much
less wall clock. That is also the control run for the last cell, which is about
what the warm start buys.

There is no `pipeline.solve_point` for this solver, so the per-point work lives
in this notebook as one picklable function (`solve_point_ks`), in the shape of
`tests/test_sweep_alpha_krylov_shur.ipynb`; only the warm-start plumbing
(`x0`, `V_prev`, and shipping `x_ritz` / `V` back) is new.

## Tolerances: fixed along the sweep, unlike the `alpha_sq` one

What the iteration has to resolve is `1 - mu` per period, which *is* the
bit-flip rate -- and the rate **grows** with `eps_p`. So the smallest `eps_p`
is the hardest point of the sweep, and one setting sized for it covers the
whole thing: loop at `1e-9 / 1e-10`, polish and Rayleigh quotient at
`1e-11 / 1e-12`, `res_tol = 1e-10`. That is band 3 of the ladder in
`tests/test_sweep_alpha_krylov_shur.ipynb` (`alpha_sq < 10`), held constant
here. The one thing that *is* adapted is in the other direction: the final
block shortens as the gap opens, since `n_blocks` beyond what resolves
`1 - mu**n` is wasted GPU time (see `n_blocks_auto`).

## Running it

1. **Kernel**: the workspace pixi environment,
   `/workspace/.pixi/envs/default/bin/python` (**Python 3.13.x**, `ray 2.56.0`)
   -- these must match the cluster.
2. **Vault session**, once (lasts about a week):

   ```bash
   export VAULT_ADDR=https://vault.int.alice-bob.com
   vault login -method=oidc role=research_core_user
   ```

3. Run all cells. The first `acr.run` of a session may open an interactive
   Microsoft device-code login for GCS read access; that is expected and cached.

If you have already run something that imports `ray` in this kernel, restart it
-- Ray caches `RAY_AUTH_MODE` at first import.

In [ ]:
import os
import subprocess
import sys

assert sys.version_info[:2] == (3, 13), (
    f"kernel runs Python {sys.version.split()[0]}; the cluster needs 3.13.x. "
    "Select the workspace pixi environment as the kernel."
)

# Ray reads RAY_AUTH_MODE once, at its first import, and caches it. Setting it
# here works *only* because nothing has imported ray yet in a fresh kernel.
os.environ["RAY_AUTH_MODE"] = "token"
os.environ.setdefault("VAULT_ADDR", "https://vault.int.alice-bob.com")

if not os.environ.get("RAY_AUTH_TOKEN"):
    try:
        os.environ["RAY_AUTH_TOKEN"] = subprocess.run(
            ["vault", "kv", "get", "-field=token", "kv-it/prod/ray/auth-token"],
            capture_output=True,
            text=True,
            check=True,
        ).stdout.strip()
    except (FileNotFoundError, subprocess.CalledProcessError) as exc:
        raise RuntimeError(
            "Could not mint the cluster token from Vault. In a terminal, run:\n"
            "  export VAULT_ADDR=https://vault.int.alice-bob.com\n"
            "  vault login -method=oidc role=research_core_user\n"
            "then re-run this cell (no kernel restart needed).\n"
            f"{(getattr(exc, 'stderr', '') or '').strip()}"
        ) from exc

import numpy as np

from anb_compute import ray as acr

import pathlib


def _find_repo():
    """Locate the Cheb_Ar checkout without hardcoding an absolute path.

    Notebooks have no __file__, so search upward from the working directory --
    which VS Code sets to either the notebook's folder or the workspace root,
    depending on `jupyter.notebookFileRoot`. Both are handled.
    """
    here = pathlib.Path.cwd().resolve()
    for base in (here, *here.parents):
        if (base / "src" / "cheb_ar").is_dir():
            return base
        if (base / "Cheb_Ar" / "src" / "cheb_ar").is_dir():
            return base / "Cheb_Ar"
    raise RuntimeError(f"no Cheb_Ar checkout found above {here}")


REPO = _find_repo()

# Shipped by PATH, not by name: a bare module name that resolves to a
# non-editable site-packages install ships a frozen copy, and local edits would
# never reach the worker. These prepend to the worker's sys.path, so the
# dynamiqs here shadows the image's stock 0.3.6 (which lacks `mesolve_fast`).
CHEB_AR = str(REPO / "src" / "cheb_ar")
# The patched dynamiqs checkout, expected beside the repo; $DYNAMIQS_SRC wins.
DYNAMIQS = os.environ.get("DYNAMIQS_SRC") or str(
    REPO.parent / "dynamiqs" / "dynamiqs"
)
for _p in (CHEB_AR, DYNAMIQS):
    if not pathlib.Path(_p).is_dir():
        raise RuntimeError(f"not a directory, cannot ship to workers: {_p}")

RUNTIME_ENV = acr.build_runtime_env(py_modules=[CHEB_AR, DYNAMIQS])

print("ready")

### Settings

Everything that does not move along the sweep. The only per-point differences
are `kappa_b` (rescaled on the worker), the restart size (`M_FIRST` cold,
`M_WARM` warm) and the length of the final block.

In [ ]:
import time

N_A, N_B = 30, 15
ALPHA_SQ = 6
EPS_P_LIST = list(np.linspace(0.1, 1.1, 6))
KAPPA_B_INIT = None  # None -> ats.KAPPA_B, read on the worker (single source of truth)
WARM_START = True

# Tolerances: sized for the *first* point, which is the hardest one, and held
# fixed (see the header). `res_tol` sits one decade above `rtol_final`, which
# is where the relative residual floors out.
TOLS = dict(
    rtol_loop=1e-9,
    atol_loop=1e-10,
    res_tol_loop=1e-8,
    rtol_final=1e-11,
    atol_final=1e-12,
    res_tol=1e-10,
    n_blocks_final=50,   # a cap; `n_blocks_auto` shortens it as the gap opens
    n_blocks_auto=True,
    rq_resolution=1e-5,  # target for 1 - mu**n_blocks, see the worker
    max_cycles=35,
)

SETTINGS = dict(n_a=N_A, n_b=N_B, alpha_sq=ALPHA_SQ, seed=0, **TOLS)

# Restart size. `m - k` is the number of period integrations per cycle and `k`
# is what is kept across a restart, so `k` must cover the slow cluster (target
# + phase-flip pair + outer bulk) whether or not the start is warm -- it is a
# property of the spectrum, not of the starting vector. A warm start buys
# *fewer cycles*, not a smaller `k`, so only `m` is trimmed for warm points.
M_FIRST = 50   # cold-started points
M_WARM = 36    # warm-started points
K_KEEP = 12

print(f"{len(EPS_P_LIST)} points, eps_p from {EPS_P_LIST[0]:.3f} to {EPS_P_LIST[-1]:.3f}, "
      f"alpha_sq = {ALPHA_SQ}, N = {N_A * N_B}")
print(f"warm start: {WARM_START}  |  m = {M_FIRST} (cold) / {M_WARM} (warm), k = {K_KEEP}")
print(f"basis on the GPU: {(M_FIRST + 1) * (N_A * N_B) ** 2 * 16 / 1024 ** 3:.2f} GB (cold cycle)")

### One point, on one GPU worker

Same body as `solve_point_ks` in `tests/test_sweep_alpha_krylov_shur.ipynb`,
plus the two things an `eps_p` chain needs: a starting vector rotated out of
the previous point's eigenbasis, and `x_ritz` / `V` shipped back so the next
point can do the same. Both are `N**2` and `N**2` complex arrays (2.4 MB each
at 32 x 12), and the coordinator drops them from the result it returns.

In [ ]:
def solve_point_ks(
    *,
    eps_p,
    alpha_sq,
    n_a,
    n_b,
    kappa_b=None,
    # Krylov-Schur
    m=50,
    k=12,
    res_tol=1e-10,
    max_cycles=35,
    # integrator tolerance for the Arnoldi loop ...
    rtol_loop=1e-9,
    atol_loop=1e-10,
    # ... switched to the final tolerance once the residual reaches this, or
    # after `max_cycles_loop` cycles (the relaxed tolerance has a residual
    # floor of its own, which `res_tol_loop` may sit below) ...
    res_tol_loop=1e-8,
    max_cycles_loop=None,
    # ... which is also what the final Rayleigh quotient uses
    rtol_final=1e-11,
    atol_final=1e-12,
    n_blocks_final=50,
    n_blocks_auto=True,
    rq_resolution=1e-5,
    # warm start
    x0=None,
    V_prev=None,
    want_x_ritz=False,
    want_V=False,
    seed=0,
    k_top=3,
):
    """One `eps_p` point with `KrylovSchurLindblad`, on one GPU worker.

    Parameters
    ----------
    kappa_b : float, optional
        Buffer loss rate; the model default (``ats.KAPPA_B``) when omitted.
        The coordinator rescales it along the sweep.
    x0, V_prev : array, optional
        Warm start: the previous point's Ritz vector and the eigenbasis it is
        written in. The interaction-frame basis moves with ``eps_p``, so the
        vector is rotated into this point's basis first -- exactly what
        ``pipeline.solve_point`` does for the `ChebAr` sweep. Both solvers
        vectorize column-major, so ``transform_vectorized_state`` applies here
        unchanged.
    want_x_ritz, want_V : bool
        Ship the Ritz vector / the eigenbasis back, which is what the next
        point of the chain needs. Off by default: they are the only large
        objects in the payload.

    Everything returned is plain numpy / python. Physics constants come from
    `cheb_ar.models.ats` (single source of truth).
    """
    import time

    import numpy as np

    from cheb_ar import KrylovSchurLindblad
    from cheb_ar.models.ats import (
        KAPPA_B,
        build_ats_hamiltonian_interaction,
        transform_vectorized_state,
    )

    def py(v):
        """0-d jax/numpy scalar -> plain python float/complex."""
        a = np.asarray(v)
        return complex(a) if np.iscomplexobj(a) else float(a)

    lines = []

    def log(s):
        print(s)
        lines.append(s)

    t0 = time.time()
    dims = (n_a, n_b)
    kappa_b = KAPPA_B if kappa_b is None else kappa_b

    # --- interaction frame, ONE drive period per block: the iteration runs on U ---
    (
        H_I,
        jump_ops_I,
        jump_ops_LdL_I,
        output_phase,
        V,
        T_block,
        params,
    ) = build_ats_hamiltonian_interaction(
        n_a=n_a,
        n_b=n_b,
        alpha_sq=alpha_sq,
        kappa_b=kappa_b,
        epsilon_p=eps_p,
        n_periods=1,
    )

    solver = KrylovSchurLindblad(
        H_I,
        jump_ops_I,
        T_block,
        jump_ops_LdL=jump_ops_LdL_I,
        dims=dims,
        output_phase=output_phase,
        rtol=rtol_loop,
        atol=atol_loop,
    )

    if x0 is None:
        x0 = solver.make_x0(seed=seed)
        warm_start = False
    else:
        if V_prev is not None:
            # The interaction-frame basis moves with eps_p, so a warm vector
            # from the previous point has to be rotated into this one.
            x0 = transform_vectorized_state(x0, np.asarray(V_prev), np.asarray(V))
        warm_start = True
    t_build = time.time() - t0

    # --- phase 1: cycles at the loop tolerance -------------------------------
    # A warm vector usually clears `res_tol_loop` in the first cycle and falls
    # straight through to the polishing phase; a cold one does not.
    same_tol = (rtol_loop, atol_loop) == (rtol_final, atol_final)
    if max_cycles_loop is None:
        max_cycles_loop = max(1, max_cycles // 2)
    t1 = time.time()
    out = solver.solve(
        x0,
        m=m,
        k=k,
        res_tol=res_tol if same_tol else res_tol_loop,
        max_cycles=max_cycles if same_tol else max_cycles_loop,
        log=log,
    )
    history = list(out["history"])
    n_cycles_loop = out["n_cycles"]
    t_loop = time.time() - t1

    # --- phase 2: polish at the final tolerance, resuming the kept subspace ---
    t2 = time.time()
    if not same_tol:
        log(f"-- switching integrator tolerance to rtol={rtol_final:g}, atol={atol_final:g}")
        solver.set_tolerance(rtol_final, atol_final)
        out = solver.solve(
            state=out["state"],
            m=m,
            k=k,
            res_tol=res_tol,
            max_cycles=max(1, max_cycles - n_cycles_loop),
            log=log,
        )
        history += list(out["history"])
    t_polish = time.time() - t2

    x_ritz = out["x_ritz"]

    # --- how long a final block to use --------------------------------------
    # The rate is read off 1 - mu**n ~ n * (1 - mu), and the relative error of
    # that reading is about rtol_final / (n * (1 - mu)) -- which is why the
    # block is long at all. Here `1 - mu` *grows* with eps_p, so a block sized
    # for the first point is waste at the last: aim for a fixed absolute
    # 1 - mu**n = `rq_resolution` instead, capped at `n_blocks_final`. With the
    # defaults that is 1e-5, i.e. six orders above rtol_final and two orders
    # above what the alpha_sq = 6 run of tests/test_krylov_schur.ipynb gets
    # away with; the cap binds at the small-eps_p end, where nothing is
    # adapted.
    gap = max(float(1 - complex(out["mu"]).real), 1e-14)
    n_blocks = (int(np.clip(int(np.ceil(rq_resolution / gap)), 1, n_blocks_final))
                if n_blocks_auto else int(n_blocks_final))
    log(f"-- final Rayleigh quotient on {n_blocks} block(s): "
        f"1 - mu = {gap:.2e} per period, 1 - mu**n ~ {n_blocks * gap:.2e} "
        f"({n_blocks * gap / rtol_final:.1e} x rtol_final)")

    # --- final numbers on the true propagator, tight tolerance ---------------
    t3 = time.time()
    res_1 = solver.residual_check(x_ritz, n_blocks=1, rtol=rtol_final, atol=atol_final)
    res_n = solver.residual_check(
        x_ritz, n_blocks=n_blocks, rtol=rtol_final, atol=atol_final
    )
    t_final = time.time() - t3

    # --- Fock weights, for the truncation check (lab frame, as in the other
    # notebooks). Only the two marginals travel, not rho itself.
    V = np.asarray(V)
    rho_lab = V @ np.asarray(solver.unvec(x_ritz)) @ V.conj().T
    D = np.einsum("ij,ij->i", rho_lab, rho_lab.conj()).real.reshape(n_a, n_b)
    w_a = D.sum(axis=1) / D.sum()
    w_b = D.sum(axis=0) / D.sum()

    import jax

    res = {
        # sweep coordinates
        "eps_p": float(eps_p),
        "alpha_sq": float(alpha_sq),
        "kappa_b": float(kappa_b),
        "n_a": int(n_a),
        "n_b": int(n_b),
        "warm_start": bool(warm_start),
        # eigenvalue from the Krylov iteration (one period) ...
        "mu": complex(out["mu"]),
        "rate_ritz": py(solver.rate_from_mu(out["mu"])),
        "res_rel": float(out["res_rel"]),
        "converged": bool(out["converged"]),
        # ... and from the final Rayleigh quotients at tight tolerance
        "mu_RQ_1": py(res_1["mu_RQ"]),
        "res_rel_1": py(res_1["res_rel"]),
        "mu_RQ_n": py(res_n["mu_RQ"]),
        "rate_bf": py(res_n["rate_RQ"]),  # named as in `pipeline.solve_point`
        "res_rel_n": py(res_n["res_rel"]),
        "n_blocks_final": int(n_blocks),
        "gap": gap,
        # cost
        "m": int(m),
        "k": int(k),
        "n_apply": int(out["n_apply"]),
        "n_cycles": len(history),
        "n_cycles_loop": int(n_cycles_loop),
        "res_hist": np.array([h["res_rel"] for h in history]),
        "apply_hist": np.array([h["n_apply"] for h in history]),
        "log": lines,
        # truncation check
        "fock_weight_a": w_a,
        "fock_weight_b": w_b,
        "tail_a": float(w_a[-k_top:].sum()),
        "tail_b": float(w_b[-k_top:].sum()),
        "k_top": int(k_top),
        "params": {kk: py(v) for kk, v in params.items()},
        "T_block": py(T_block),
        "settings": {
            "n_a": n_a, "n_b": n_b, "m": m, "k": k,
            "res_tol": res_tol, "res_tol_loop": res_tol_loop,
            "max_cycles": max_cycles, "max_cycles_loop": max_cycles_loop,
            "rtol_loop": rtol_loop, "atol_loop": atol_loop,
            "rtol_final": rtol_final, "atol_final": atol_final,
            "n_blocks_final": n_blocks_final, "n_blocks_auto": n_blocks_auto,
            "rq_resolution": rq_resolution, "seed": seed,
        },
        "timings_s": {
            "build": t_build,
            "loop": t_loop,
            "polish": t_polish,
            "final_RQ": t_final,
            "total": time.time() - t0,
        },
        "device": str(jax.devices()[0]),
    }
    if want_x_ritz:
        res["x_ritz"] = np.asarray(x_ritz)
    if want_V:
        res["V"] = V
    return res


#: Coordinates copied onto a failed point, so it still says where it was.
_COORD_KEYS = ("eps_p", "alpha_sq", "kappa_b", "n_a", "n_b", "m", "k", "res_tol")


def solve_point_ks_safe(**kwargs):
    """`solve_point_ks`, returning a failure instead of raising it.

    Same contract as `cheb_ar.pipeline.solve_point_safe`. In a warm chain a
    failure also costs the *next* point its starting vector, so the coordinator
    drops back to a cold start after one.
    """
    try:
        return solve_point_ks(**kwargs)
    except Exception as exc:  # noqa: BLE001 - one bad point must not sink the sweep
        entry = {k: kwargs.get(k) for k in _COORD_KEYS}
        entry["error"] = f"{type(exc).__name__}: {exc}"
        return entry

### The chain

In [ ]:
def sweep(worker, eps_p_list, settings, kappa_b_init, m_first, m_warm, k_keep,
          warm_start):
    """Coordinator on a worker; one GPU task per point.

    Sequential when `warm_start`: point i+1 needs point i's x_ritz *and* its V
    (the eigenbasis to rotate that vector out of). Otherwise all points are
    submitted at once. `worker` is passed in rather than imported: it is
    defined in this notebook, so it ships by value with this function.
    """
    import numpy as np
    import ray

    from anb_compute import ray as acr
    from cheb_ar.models.ats import KAPPA_B

    kappa_b_init = KAPPA_B if kappa_b_init is None else kappa_b_init
    eps_p_init = eps_p_list[0]

    def kappa_b_for(e):
        return float(np.sin(e) / np.sin(eps_p_init) * kappa_b_init)

    out = []
    if not warm_start:
        refs = {
            acr.submit(worker, eps_p=float(e), kappa_b=kappa_b_for(e),
                       m=m_first, k=k_keep,
                       num_gpus=1, num_cpus=8, **settings): float(e)
            for e in eps_p_list
        }
        pending = list(refs)
        while pending:
            ready, pending = ray.wait(pending, num_returns=1)
            ref, e = ready[0], refs[ready[0]]
            try:
                res = ray.get(ref)
            except Exception as exc:
                out.append({"eps_p": e, "error": f"{type(exc).__name__}: {exc}"})
                continue
            out.append(res)
            print(f"eps_p = {e:5.3f}  "
                  + (res["error"] if "error" in res
                     else f"rate_bf = {res['rate_bf'].real:.6e}"
                          f"  apps = {res['n_apply']:d}"
                          f"  cycles = {res['n_cycles']:d}"))
    else:
        prev = None
        for e in eps_p_list:
            warm = prev is not None
            ref = acr.submit(
                worker,
                eps_p=float(e), kappa_b=kappa_b_for(e),
                m=m_warm if warm else m_first, k=k_keep,
                x0=prev["x_ritz"] if warm else None,
                V_prev=prev["V"] if warm else None,
                want_x_ritz=True, want_V=True,
                num_gpus=1, num_cpus=8, **settings,
            )
            try:
                res = ray.get(ref)
            except Exception as exc:
                out.append({"eps_p": float(e),
                            "error": f"{type(exc).__name__}: {exc}"})
                prev = None  # chain broken, next point restarts cold
                continue
            print(f"eps_p = {e:5.3f}  "
                  + (res["error"] if "error" in res
                     else f"rate_bf = {res['rate_bf'].real:.6e}"
                          f"  apps = {res['n_apply']:d}"
                          f"  cycles = {res['n_cycles']:d}"
                          f"  (warm={res['warm_start']})"))
            prev = res if "error" not in res else None
            # V and x_ritz only existed to feed the next point.
            out.append({k: v for k, v in res.items() if k not in ("V", "x_ritz")})

    out.sort(key=lambda d: d["eps_p"])
    return out


t0 = time.time()
results = acr.run(
    sweep, solve_point_ks_safe, EPS_P_LIST, SETTINGS, KAPPA_B_INIT,
    M_FIRST, M_WARM, K_KEEP, WARM_START,
    runtime_env=RUNTIME_ENV,
    poll_seconds=5,
)
print(f"\nwall clock, incl. job startup: {time.time() - t0:.1f} s "
      f"for {len(results)} point(s)")

### Results

In [ ]:
ok = [r for r in results if "error" not in r]
bad = [r for r in results if "error" in r]

eps_p = np.array([r["eps_p"] for r in ok])
bf_list = np.array([r["rate_bf"] for r in ok])
kappa_b_list = np.array([r["kappa_b"] for r in ok])

print(f"{'eps_p':>7} {'kappa_b':>9} {'rate_bf':>14} {'1-Re mu':>10} "
      f"{'res Krylov':>11} {'res RQ':>9} {'blocks':>7} {'1-mu^n':>9} {'/rtol':>8} "
      f"{'apps':>6} {'cyc':>4} {'warm':>5} {'conv':>5} {'GPU s':>7}")
for r in ok:
    s = r["settings"]
    drop = r["n_blocks_final"] * r["gap"]
    print(f"{r['eps_p']:7.3f} {r['kappa_b']:9.5f} {r['rate_bf'].real:14.6e} "
          f"{r['gap']:10.2e} {r['res_rel']:11.2e} {r['res_rel_n']:9.2e} "
          f"{r['n_blocks_final']:7d} {drop:9.2e} {drop / s['rtol_final']:8.1e} "
          f"{r['n_apply']:6d} {r['n_cycles']:4d} {str(r['warm_start']):>5} "
          f"{('yes' if r['converged'] else 'NO'):>5} "
          f"{r['timings_s']['loop'] + r['timings_s']['polish']:7.1f}")
for r in bad:
    print(f"{r['eps_p']:7.3f}  FAILED: {r['error']}")

stalled = [r for r in ok if not r["converged"]]
if stalled:
    print("\nnot converged (residual above res_tol at the cycle budget): "
          + ", ".join(f"eps_p={r['eps_p']:.3f} (res {r['res_rel']:.1e} "
                      f"> {r['settings']['res_tol']:.0e})" for r in stalled))

# The claim the fixed tolerances rest on: the gap grows along the sweep, so the
# first point is the hardest one. If this prints a decrease, the setting was
# sized for the wrong end.
if len(ok) > 1:
    print(f"\ngap 1 - Re mu: {ok[0]['gap']:.2e} at eps_p = {ok[0]['eps_p']:.3f} "
          f"-> {ok[-1]['gap']:.2e} at eps_p = {ok[-1]['eps_p']:.3f} "
          f"({'growing, as assumed' if ok[-1]['gap'] > ok[0]['gap'] else 'SHRINKING -- re-size the tolerances for the last point'})")

### Bit-flip rate vs pump strength

In [ ]:
import matplotlib.pyplot as plt

converged = np.array([r["converged"] for r in ok])

fig, ax = plt.subplots(figsize=(5.5, 4.0), constrained_layout=True)
ax.scatter(eps_p[converged], bf_list.real[converged], zorder=3, label="Krylov-Schur")
if (~converged).any():
    ax.scatter(eps_p[~converged], bf_list.real[~converged], facecolors="none",
               edgecolors="#eb6834", zorder=3, label="not converged")
    ax.legend()
ax.set_yscale("log")
ax.set_xlabel(r"$\varepsilon_p$")
ax.set_ylabel("bit-flip rate (GHz)")
ax.grid(alpha=0.3)
plt.show()

### Adiabatic ratio

In [ ]:
# Adiabaticity check: kappa_b / g_2 should stay roughly constant by
# construction (that is what the kappa_b rescaling along the sweep is for).
# g2 comes back in each point's `params`, so the constants stay in ats.py; the
# /2pi*1e3 is the rad-per-ns -> MHz conversion the original notebook applied.
g_2 = np.array([r["params"]["g2"] for r in ok]) / 2 / np.pi * 1e3
ratio = kappa_b_list / g_2

fig, ax = plt.subplots(figsize=(5.5, 3.4), constrained_layout=True)
ax.plot(eps_p, ratio, "o-")
ax.set_xlabel(r"$\varepsilon_p$")
ax.set_ylabel(r"$\kappa_b / g_2$")
ax.set_title("adiabatic ratio along the sweep")
ax.grid(alpha=0.3)
plt.show()

print("kappa_b / g_2 :", np.round(ratio, 4))

### What the warm start bought

Cost per point, and the residual trace behind it. The first point is cold; if
the chain is working, every later point starts a decade or more below where
the cold one started and converges in fewer cycles. Run the whole notebook
again with `WARM_START = False` for the control -- the numbers to compare are
period integrations (`n_apply`) and GPU seconds per point, not wall clock,
which the cold run wins by fanning out.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl

fig, axes = plt.subplots(1, 2, figsize=(10.4, 3.9), constrained_layout=True)

cmap = mpl.colormaps["viridis"]
norm = mpl.colors.Normalize(vmin=eps_p.min(), vmax=eps_p.max())
for r in ok:
    c = cmap(norm(r["eps_p"]))
    axes[0].plot(r["apply_hist"], r["res_hist"], marker=".", ms=4, lw=1.1, color=c,
                 ls="-" if r["warm_start"] else "--")
axes[0].axhline(ok[0]["settings"]["res_tol"], color="0.4", lw=0.8, ls=":",
                label="res_tol")
axes[0].set_yscale("log")
axes[0].set_xlabel("period integrations")
axes[0].set_ylabel(r"residual $\|U y - \theta y\| / |\theta|$")
axes[0].set_title("target residual per cycle (dashed: cold start)", fontsize=9)
axes[0].legend(fontsize=8, frameon=False)
fig.colorbar(mpl.cm.ScalarMappable(norm=norm, cmap=cmap), ax=axes[0],
             label=r"$\varepsilon_p$")

apps = np.array([r["n_apply"] for r in ok])
warm = np.array([r["warm_start"] for r in ok])
t_loop = np.array([r["timings_s"]["loop"] for r in ok])
t_pol = np.array([r["timings_s"]["polish"] for r in ok])
t_rq = np.array([r["timings_s"]["final_RQ"] for r in ok])
w = 0.6 * np.min(np.diff(np.sort(eps_p))) if len(eps_p) > 1 else 0.1
axes[1].bar(eps_p, t_loop, width=w, label="loop (relaxed)", color="#2a78d6")
axes[1].bar(eps_p, t_pol, width=w, bottom=t_loop, label="polish (tight)",
            color="#eb6834")
axes[1].bar(eps_p, t_rq, width=w, bottom=t_loop + t_pol,
            label="final Rayleigh quotient", color="#8c8c8c")
for x, y, a_ in zip(eps_p, t_loop + t_pol + t_rq, apps):
    axes[1].annotate(f"{a_}", (x, y), textcoords="offset points", xytext=(0, 2),
                     ha="center", fontsize=6, color="0.3")
axes[1].set_xlabel(r"$\varepsilon_p$")
axes[1].set_ylabel("GPU time (s)")
axes[1].set_title("cost per point (label: period integrations)", fontsize=9)
axes[1].legend(fontsize=8, frameon=False)

for ax in axes:
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.grid(True, which="major", lw=0.4, color="0.9")
    ax.set_axisbelow(True)
plt.show()

if warm.any() and (~warm).any():
    print(f"cold points: {apps[~warm].mean():.0f} period integrations on average "
          f"({np.asarray([r['n_cycles'] for r in ok])[~warm].mean():.1f} cycles)")
    print(f"warm points: {apps[warm].mean():.0f} period integrations on average "
          f"({np.asarray([r['n_cycles'] for r in ok])[warm].mean():.1f} cycles)")
    print(f"saving on a warm point: {1 - apps[warm].mean() / apps[~warm].mean():+.0%}")
else:
    print(f"all points {'warm' if warm.all() else 'cold'}: "
          f"{apps.mean():.0f} period integrations on average. Re-run with "
          f"WARM_START = {not warm.all()} for the comparison.")

### Truncation check

Fixed truncation along this sweep -- `alpha_sq` does not move, so the memory
occupation does not either, and the buffer's `(epsilon_d / kappa_b)**2` is held
roughly constant by the `kappa_b` rescaling. That is the expectation; this is
the check. The weight in the top `k_top` levels must land far below the
residual being quoted.

In [ ]:
TOL = 1e-11  # same absolute criterion as the single-point notebooks

print(f"{'eps_p':>7} {'n_a':>4} {'n_b':>4} {'tail_a':>10} {'tail_b':>10} "
      f"{'res RQ':>10} {'1-|mu|':>10} {'verdict':>8}")
worst = []
for r in ok:
    passed = max(r["tail_a"], r["tail_b"]) < TOL
    if not passed:
        worst.append(r)
    print(f"{r['eps_p']:7.3f} {r['n_a']:4d} {r['n_b']:4d} "
          f"{r['tail_a']:10.2e} {r['tail_b']:10.2e} {r['res_rel_n']:10.2e} "
          f"{1 - abs(r['mu']):10.2e} {('PASS' if passed else 'FAIL'):>8}")
print(f"\ntop-{ok[0]['k_top']} weight against tol = {TOL:.0e}: "
      + ("all points PASS" if not worst else
         "raise N_A / N_B for eps_p = "
         + ", ".join(f"{r['eps_p']:.3f}" for r in worst)))

for r in (ok[0], ok[-1]):
    print(f"\neps_p = {r['eps_p']:.3f}  (n_a = {r['n_a']}, n_b = {r['n_b']}, "
          f"kappa_b = {r['kappa_b']:.5f})")
    for label, w_ in (("a (storage)", r["fock_weight_a"]), ("b (buffer)", r["fock_weight_b"])):
        top = " ".join(f"{v:.1e}" for v in np.asarray(w_)[-4:])
        print(f"  mode {label:<12} last 4 weights: {top}")